# 12 MLA from Intuition to Implementation

## Problem

当 GQA 仍然不够省时，MLA 想解决什么问题？它和普通 KV cache、MQA、GQA 最关键的区别到底在哪里？

## Dependencies

建议先完成 `10_kv_cache_and_inference.ipynb` 和 `11_mqa_and_gqa.ipynb`。


## Goals

- 理解 MLA 的核心动机：进一步降低注意力状态缓存成本
- 理解 latent compression 的基本直觉
- 理解为什么 MLA 不是“另一个名字的 GQA”
- 能通过一个最小实现看清“先压缩存储，再映射回计算空间”的思路

## Scope and Approach

这一节不追求复刻某个工业仓库的全部细节，而是要把 MLA 最值得学的那层思想拆出来：如果历史 K/V 太占地方，我们能不能不直接存原始 K/V，而是存一个更紧凑的 latent 状态，等要计算注意力时再把它映射回来？


## 先把问题重新说一遍

GQA 已经在减少 K/V head 数量，但只要：

- 上下文很长
- 层数很多
- batch 很大

KV cache 仍然会吃掉大量显存和带宽。于是问题会继续往前推：

**我们能不能不直接缓存那么多原始 K/V，而是缓存一个更紧凑的状态？**

这就是理解 MLA 的入口。


## MLA 和 GQA 到底差在哪

这一点必须说清，不然很容易把 MLA 误读成“另一种共享 K/V 的方式”。

- GQA：主要是在 **head 份数** 上做共享
- MLA：更进一步，在 **被缓存的状态表示形式** 上做压缩

也就是说，GQA 更像在问：

- “我能不能少存几组 K/V head？”

而 MLA 更像在问：

- “我能不能不直接存完整 K/V，而是存一个更小的 latent 表示？”

这就是两者最重要的层级差别。


In [ ]:
import numpy as np

np.random.seed(123)
np.set_printoptions(precision=3, suppress=True)

seq_len = 4
hidden_dim = 8
latent_dim = 3
head_dim = 4
num_heads = 2

x = np.random.randn(seq_len, hidden_dim) * 0.2

# 先把每个位置压到更小的 latent 空间里。
W_latent = np.random.randn(hidden_dim, latent_dim) * 0.3

# 再从 latent 状态映射回 K/V 所需的空间。
W_k_up = np.random.randn(latent_dim, num_heads * head_dim) * 0.3
W_v_up = np.random.randn(latent_dim, num_heads * head_dim) * 0.3

# Q 仍然可以从当前输入表示直接产生。
W_q = np.random.randn(hidden_dim, num_heads * head_dim) * 0.3


## 一个最小 MLA 视角：先存 latent，再还原 K/V

这里我们故意把事情做得非常朴素，好让思路露出来：

1. 输入 `x` 先被压到更小的 latent 空间
2. 真正需要 attention 时，再把 latent 映射回 K/V 所需空间
3. 当前 token 的 Q 再和这些还原出来的 K 做匹配

你可以把它理解成：

- 先存压缩版历史状态
- 用的时候再解压到计算所需空间


In [ ]:
# 第一步：把每个 token 压到更小的 latent 空间。
latent_cache = x @ W_latent
print('latent_cache.shape =', latent_cache.shape)  # (seq_len, latent_dim)

# 第二步：需要做 attention 时，再把 latent 映射回 K/V 计算空间。
K = (latent_cache @ W_k_up).reshape(seq_len, num_heads, head_dim)
V = (latent_cache @ W_v_up).reshape(seq_len, num_heads, head_dim)
Q = (x @ W_q).reshape(seq_len, num_heads, head_dim)

print('Q.shape =', Q.shape)
print('K.shape =', K.shape)
print('V.shape =', V.shape)


## 为什么它可能更省

如果直接缓存原始 K/V，历史每个位置都要存：

- `num_heads * head_dim` 的 K
- `num_heads * head_dim` 的 V

而如果只缓存 latent，那么每个位置只需要存 `latent_dim` 个数。只要 `latent_dim` 明显小于原始 K/V 的总维度，就有机会省下很多缓存体积。

当然，这不是白送的，因为你之后还要做“从 latent 映射回 K/V 计算空间”的额外过程。


In [ ]:
raw_kv_cache_size = seq_len * num_heads * head_dim * 2
latent_cache_size = seq_len * latent_dim

print('raw_kv_cache_size =', raw_kv_cache_size)
print('latent_cache_size =', latent_cache_size)


In [ ]:
def softmax(logits):
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / np.sum(exp, axis=-1, keepdims=True)

outputs = []
for h in range(num_heads):
    q = Q[:, h, :]
    k = K[:, h, :]
    v = V[:, h, :]

    scores = (q @ k.T) / np.sqrt(head_dim)
    mask = np.triu(np.ones_like(scores), k=1) * -1e9
    weights = softmax(scores + mask)
    outputs.append(weights @ v)

mla_output = np.concatenate(outputs, axis=-1)
print('mla_output.shape =', mla_output.shape)
print(mla_output)


## 这一节真正要学会的不是公式，而是层级

很多人会在这里被细节淹掉，结果忘了真正要抓住的主线。理解 MLA 最重要的是知道它在哪个层级上做文章：

- MHA / MQA / GQA：主要在“head 组织方式”上做变化
- MLA：进一步在“被缓存的历史状态表示”上做变化

所以 MLA 值得学，不是因为它换了个新名字，而是因为它代表一种更深入的推理优化思路：

**不仅共享历史状态，还压缩历史状态。**

## Common mistakes

- 把 MLA 误解成“又一种换名字的 GQA”。它更关键的点在于缓存表示的压缩与重构思路。
- 一上来就纠结工业实现细节，结果反而看不清核心思想。
- 只关注省显存，不去想从 latent 再映射回计算空间带来的额外代价。
- 以为 MLA 只是“低秩近似”的另一种说法，而忽略它在推理 cache 结构上的工程动机。

## Checkpoints

- 用自己的话解释 MLA 和 GQA 的主要差别。
- 把 `latent_dim` 改成 2、4、6，观察 cache 大小和表示空间之间的变化。
- 回答：MLA 主要是在减少 query head 数量，还是在减少被缓存状态的体积？
- 说明为什么“压缩得越狠越好”并不成立。

## Summary

MLA 的核心价值在于，它把大模型推理里的一个现实问题摆得很清楚：瓶颈不只来自算力，也来自状态存储和带宽。和 GQA 相比，MLA 更进一步，不只是共享历史状态，而是尝试把历史状态本身压缩成更紧凑的 latent 形式，再在需要时映射回计算空间。

## Next Module

下一节进入 MoE。它对应的是另一条完全不同但同样关键的现代模型路线：不是省 cache，而是用稀疏激活把总容量做大。
